# Sensitivity — Hand-Added Labeled Addresses

The labeled-address list includes 44 addresses added by hand while the list was compiled
(`data/20241214_labeled_addresses/readme.txt`). Labeling a gas provider changes the features of
every wallet it funded (`provider_is_labeled`, `provider_is_star_like_attack`, chain and tree
features) and their funding groups. If Sybil labels had influenced which addresses were added,
that would be label leakage through feature construction.

20 of the 44 are also in the hildobby CEX list, a public source, so their labels never depended on
the hand step. This notebook measures the effect of the other 24 ("hand-only"): it rebuilds every
feature, the taxonomy, and the funding groups **without** them, re-splits with the group split, retrains
the selected LightGBM configuration (3 seeds), and compares test metrics with the standard
pipeline. Thresholds are chosen on each version's own validation set, as in `02`.

In [ ]:
import os, re, time, json, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier
import lightgbm as lgb_lib
import sybil_pipeline as sp
warnings.filterwarnings('ignore')

DATA_DIR, SPLIT_METHOD, SEED = './data', 'group', sp.SEED
P = sp.data_paths(DATA_DIR)
readme = open(os.path.join(DATA_DIR, '20241214_labeled_addresses', 'readme.txt')).read()
HAND_ADDED = sorted({a.lower() for a in re.findall(r"labeled_addresses\.add\('(0x[0-9a-fA-F]{40})'", readme)})
# 20 of them are also in the hildobby CEX list (a public source), so labeling them never depended
# on the hand step. Only the remainder is tested.
HILDOBBY = set(pd.read_csv(os.path.join(DATA_DIR, '20241013_hildobby_cex_evms',
                                        'All_Known_EVM_CEX_Addresses.2024-10-13.csv'))['address'].str.lower())
HAND_ONLY = sorted(set(HAND_ADDED) - HILDOBBY)
print(f'Hand-added labeled addresses: {len(HAND_ADDED)}  |  also in hildobby: {len(HAND_ADDED) - len(HAND_ONLY)}'
      f'  |  hand-only (tested here): {len(HAND_ONLY)}')

In [ ]:
def build(drop_labels=()):
    """build_master_df with an optional set of addresses removed from the labeled list."""
    df = sp.load_l0(P['l0'])
    funding = sp.load_provision(P['gas_prov'])
    tfm = sp.provision_map(funding)
    anchors = sp.stream_labeled_anchors(P['labeled'], set(tfm) | set(tfm.values())) - set(drop_labels)
    df = sp.add_provider_flags(df, tfm, anchors)
    df = sp.merge_provision_features(df, sp.provision_features(funding, set(df['addr']), anchors))
    df = sp.merge_cex_dex(df, P['cex'])
    df = sp.add_labels(df, P['sybil'], tfm)
    df = sp.add_chain_features(df, tfm, anchors)
    df = sp.add_category(df)
    df = sp.add_funding_group(df, tfm, anchors)
    assert df['addr'].is_unique
    return df, anchors

base, anchors_base = build()
alt, anchors_alt = build(HAND_ONLY)
print(f'Labeled addresses in network: standard {len(anchors_base):,}, without hand-only {len(anchors_alt):,}')
print(f'Hand-only addresses present in the network: {len(anchors_base - anchors_alt)}')

## Wallets whose features change

In [ ]:
b, a = base.set_index('addr'), alt.set_index('addr').loc[base['addr']]
changed = pd.Series(False, index=b.index)
for f in sp.FEATS:
    changed |= ~np.isclose(b[f].astype(float), a[f].astype(float), rtol=1e-9, atol=1e-12)
print(f'Wallets with any feature changed: {changed.sum():,} of {len(b):,}  '
      f'(Sybil: {int(b.loc[changed, "sybil"].sum())}, i.e. {b.loc[changed, "sybil"].mean()*100:.2f} %)')
print('\nCategory of changed wallets (standard -> without hand-added):')
print(pd.crosstab(b.loc[changed, 'category'], a.loc[changed, 'category']).to_string())
print(f'\nFunding groups: standard {base.funding_group.nunique():,}, without hand-only {alt.funding_group.nunique():,}')

## LightGBM on each version (group split, selected configuration)

In [ ]:
_, LGBM_SELECTED = sp.load_selected_params()
LGBM_PARAMS = dict(n_estimators=5000, subsample_freq=1, colsample_bytree=0.8,
                   verbose=-1, n_jobs=sp.N_JOBS, **LGBM_SELECTED)
SEEDS = [42, 123, 456]

def evaluate(df, label):
    S = sp.make_splits(df, sp.FEATS, method=SPLIT_METHOD, seed=SEED)
    val, test = [], []
    for seed in SEEDS:
        m = LGBMClassifier(**LGBM_PARAMS, random_state=seed)
        m.fit(S['X_train'], S['y_train'], eval_set=[(S['X_val'], S['y_val'])],
              callbacks=[lgb_lib.early_stopping(50, verbose=False), lgb_lib.log_evaluation(-1)])
        val.append(m.predict_proba(S['X_val'])[:, 1]); test.append(m.predict_proba(S['X_test'])[:, 1])
    pv, pt = np.mean(val, axis=0), np.mean(test, axis=0)
    thr = sp.best_f1_threshold(S['y_val'], pv)
    yhat = (pt >= thr).astype(int); y = S['y_test']
    r = dict(name=label, threshold=thr, precision=precision_score(y, yhat), recall=recall_score(y, yhat),
             f1=f1_score(y, yhat), auroc=roc_auc_score(y, pt), ap=average_precision_score(y, pt),
             test_rows=len(y), test_sybils=int(y.sum()))
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()})
    return r

t0 = time.time()
res = [evaluate(base, 'standard labeled list'), evaluate(alt, 'without 24 hand-only labels')]
print(f'({time.time()-t0:.0f}s)')
cmp = pd.DataFrame(res).set_index('name')[['precision', 'recall', 'f1', 'auroc', 'ap', 'threshold']]
cmp.loc['difference'] = cmp.iloc[1] - cmp.iloc[0]
print(cmp.round(4).to_string())

In [ ]:
sp.save_results(res, '07_sensitivity_hand_labels', SPLIT_METHOD, extra=dict(
    n_hand_added=len(HAND_ADDED), n_hand_only=len(HAND_ONLY), n_hand_added_in_network=len(anchors_base - anchors_alt),
    wallets_changed=int(changed.sum()), sybils_changed=int(b.loc[changed, 'sybil'].sum()),
    params=LGBM_SELECTED))